# NB18 — Why Fast-DetectGPT failed: is it a length confound or the surrogate?

**CPU only. Internet ON** (one tokenizer download). Runs in a few minutes.

Fast-DetectGPT gave **AUC 50.44 on our corpus** (chance) but **AUC 35.34 on Ar-APT** — *below* chance,
with the class means reversed (ours: human −5.085, AI −3.756; Ar-APT: human −4.710, AI −7.351). A
statistic that flips sign between datasets is not measuring authorship consistently, and the thesis
needs a mechanism, not a shrug.

**The suspicion.** The statistic is
`d(x) = (log p(x) − Σμ_j) / sqrt(Σσ²_j)`. The numerator is a **sum** over positions (grows ~n), the
denominator a **square root of a sum** (grows ~√n), so `d` scales roughly with **√n**. It is *not*
length-normalised. Our articles were truncated at 1024 tokens; Ar-APT articles are ~183 words and mostly
far below that cap.

**Two hypotheses this notebook separates:**
- **H1 — length confound:** `d` tracks article length rather than authorship. Predicts a strong
  correlation between `d` and effective length, and that length *alone* is about as predictive as `d`.
- **H2 — surrogate failure:** AraGPT2 simply cannot distinguish these generators. Predicts weak
  correlation with length, and that removing the length component leaves `d` still at chance.

Both are reportable; they just require different sentences in the thesis.

---

### Inputs — what each one is and where it comes from

| variable | what it is | where to get it |
|---|---|---|
| `P_FD_OURS` | Fast-DetectGPT score per article on **our** corpus (`article_id, label, split, fastdetect`) | **NB17** output → `nb17_fastdetectgpt.parquet` |
| `P_FD_ARAPT` | the same statistic on **Ar-APT**'s balanced 400/400 set (`arapt_id, label, fastdetect`) | **NB17** output → `nb17_fastdetectgpt_arapt.parquet` (written by the sanity-check block) |
| `P_DATASET` | our 7,101-article corpus, needed for the article **text** to measure length | dataset **aigt-dataset** → `dataset.parquet` |
| `P_ARAPT` | the prepared Ar-APT table, needed for its **text** | **NB16** output → `arapt_prepared.parquet` |
| `SCORER_ID` | the AraGPT2 tokenizer, so length is counted in the **same tokens** the scorer saw | downloaded from HuggingFace automatically (needs Internet) |

All four paths are auto-discovered by filename across every attached dataset, so a wrong folder name
will not break the run — but the datasets themselves must be **attached as inputs**.

## 1 · Config and input discovery

In [1]:
import os, glob, numpy as np, pandas as pd
FD_MAXLEN = 1024                      # the truncation NB17 used when scoring
SCORER_ID = "aubmindlab/aragpt2-base"
OUT = "/kaggle/working"

def find(fname, hint):
    hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    print(f"[1/6] {fname:38s} -> {hits[0] if hits else 'NOT FOUND  (' + hint + ')'}", flush=True)
    return hits[0] if hits else None

P_FD_OURS  = "/kaggle/input/datasets/bahaaqassem/nb17-dataset/nb17_fastdetectgpt.parquet"
P_FD_ARAPT = "/kaggle/input/datasets/bahaaqassem/nb17-dataset/nb17_fastdetectgpt_arapt.parquet"
P_DATASET  = "/kaggle/input/datasets/bahaaqassem/aig-and-humang-dataset/dataset.parquet"
P_ARAPT    = "/kaggle/input/notebooks/bahaaqassem/nb16-prep-bootstrap/arapt_prepared.parquet"
assert P_FD_OURS and P_DATASET, "the two mandatory inputs are missing — see the table above"
print("[1/6] config ready", flush=True)

[1/6] config ready


## 2 · Measure length in the scorer's own tokens (capped at the truncation limit)

In [2]:
!pip install -q transformers
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(SCORER_ID)
print(f"[2/6] tokenizer {SCORER_ID} loaded", flush=True)

def eff_len(texts, tag):
    # number of tokens the scorer actually saw = min(len, FD_MAXLEN)
    out = np.zeros(len(texts), np.int32)
    for i, t in enumerate(texts):
        out[i] = min(len(tok(str(t), add_special_tokens=False)["input_ids"]), FD_MAXLEN)
        if (i+1) % 1000 == 0: print(f"[2/6]   {tag} {i+1}/{len(texts)}", flush=True)
    return out

FD = pd.read_parquet(P_FD_OURS)
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
FD["words"]  = df.loc[FD.article_id, "text"].astype(str).str.split().str.len().to_numpy()
FD["ntok"]   = eff_len(df.loc[FD.article_id, "text"].astype(str).tolist(), "ours")
FD["capped"] = FD.ntok >= FD_MAXLEN
FD = FD.dropna(subset=["fastdetect"])
print(f"\n[2/6] OUR corpus: {len(FD)} scored | median {int(FD.words.median())} words, "
      f"{int(FD.ntok.median())} tokens | hit the {FD_MAXLEN}-token cap: "
      f"{100*FD.capped.mean():.1f}%", flush=True)
print(f"[2/6]   words  human {FD[FD.label==0].words.median():.0f} | AI {FD[FD.label==1].words.median():.0f}", flush=True)
print(f"[2/6]   tokens human {FD[FD.label==0].ntok.median():.0f} | AI {FD[FD.label==1].ntok.median():.0f}", flush=True)
print(f"[2/6]   capped human {100*FD[FD.label==0].capped.mean():.1f}% | AI {100*FD[FD.label==1].capped.mean():.1f}%", flush=True)

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[2/6] tokenizer aubmindlab/aragpt2-base loaded
[2/6]   ours 1000/7101
[2/6]   ours 2000/7101
[2/6]   ours 3000/7101
[2/6]   ours 4000/7101
[2/6]   ours 5000/7101
[2/6]   ours 6000/7101
[2/6]   ours 7000/7101

[2/6] OUR corpus: 7101 scored | median 589 words, 790 tokens | hit the 1024-token cap: 29.4%
[2/6]   words  human 592 | AI 587
[2/6]   tokens human 801 | AI 779
[2/6]   capped human 30.3% | AI 28.7%


## 3 · The decisive test

Three numbers decide between H1 and H2:

1. **corr(d, √n)** — how much of `d` is just length.
2. **AUC of length alone** — if length by itself is as predictive as `d`, `d` is measuring length.
3. **AUC of the length-residual of d** — regress `d` on `√n`, keep the residual, re-score. If the AUC
   jumps, length was masking a real signal; if it stays at chance, there was no signal to mask.

In [3]:
from sklearn.metrics import roc_auc_score
from scipy.stats import pearsonr, spearmanr

def analyse(D, tag):
    d, n, ylab = D.fastdetect.to_numpy(float), D.ntok.to_numpy(float), D.label.to_numpy()
    sq = np.sqrt(n)
    rp = pearsonr(d, sq); rs = spearmanr(d, n)
    auc_d   = 100*roc_auc_score(ylab, d)
    auc_len = 100*roc_auc_score(ylab, n)
    b, a = np.polyfit(sq, d, 1)                      # d ≈ a + b*sqrt(n)
    resid = d - (a + b*sq)
    auc_r   = 100*roc_auc_score(ylab, resid)
    r2 = 1 - resid.var()/d.var()
    print(f"\n[3/6] === {tag} (n={len(D)}) ===", flush=True)
    print(f"[3/6]   corr(d, sqrt(len))     pearson {rp[0]:+.3f} (p={rp[1]:.1e}) | spearman {rs[0]:+.3f}", flush=True)
    print(f"[3/6]   variance of d explained by length: {100*r2:.1f}%", flush=True)
    print(f"[3/6]   AUC  d (raw)                = {auc_d:6.2f}", flush=True)
    print(f"[3/6]   AUC  length alone           = {auc_len:6.2f}", flush=True)
    print(f"[3/6]   AUC  d after removing length= {auc_r:6.2f}   (change {auc_r-auc_d:+.2f})", flush=True)
    print(f"[3/6]   class means of d: human {d[ylab==0].mean():+.3f} | AI {d[ylab==1].mean():+.3f}", flush=True)
    print(f"[3/6]   class median tokens: human {np.median(n[ylab==0]):.0f} | AI {np.median(n[ylab==1]):.0f}", flush=True)
    return {"corpus":tag, "n":len(D), "corr_sqrt":round(float(rp[0]),3),
            "var_expl_pct":round(100*float(r2),1), "auc_d":round(auc_d,2),
            "auc_len":round(auc_len,2), "auc_resid":round(auc_r,2),
            "mean_d_human":round(float(d[ylab==0].mean()),3),
            "mean_d_ai":round(float(d[ylab==1].mean()),3),
            "med_tok_human":float(np.median(n[ylab==0])), "med_tok_ai":float(np.median(n[ylab==1]))}

rows = [analyse(FD, "OUR corpus (all splits)"),
        analyse(FD[FD.split=="test"], "OUR corpus (test split)")]


[3/6] === OUR corpus (all splits) (n=7101) ===
[3/6]   corr(d, sqrt(len))     pearson -0.185 (p=1.7e-55) | spearman -0.188
[3/6]   variance of d explained by length: 3.4%
[3/6]   AUC  d (raw)                =  51.64
[3/6]   AUC  length alone           =  47.97
[3/6]   AUC  d after removing length=  51.38   (change -0.27)
[3/6]   class means of d: human -5.085 | AI -3.756
[3/6]   class median tokens: human 801 | AI 779

[3/6] === OUR corpus (test split) (n=1093) ===
[3/6]   corr(d, sqrt(len))     pearson -0.195 (p=8.1e-11) | spearman -0.169
[3/6]   variance of d explained by length: 3.8%
[3/6]   AUC  d (raw)                =  50.44
[3/6]   AUC  length alone           =  48.84
[3/6]   AUC  d after removing length=  50.15   (change -0.28)
[3/6]   class means of d: human -4.875 | AI -3.372
[3/6]   class median tokens: human 759 | AI 756


## 4 · Same analysis on Ar-APT, where the sign reversed

In [4]:
if P_FD_ARAPT and P_ARAPT:
    FA = pd.read_parquet(P_FD_ARAPT)
    AR = pd.read_parquet(P_ARAPT).set_index("arapt_id")
    FA["words"] = AR.loc[FA.arapt_id, "text"].astype(str).str.split().str.len().to_numpy()
    FA["ntok"]  = eff_len(AR.loc[FA.arapt_id, "text"].astype(str).tolist(), "arapt")
    FA = FA.dropna(subset=["fastdetect"])
    print(f"\n[4/6] Ar-APT: {len(FA)} scored | median {int(FA.words.median())} words, "
          f"{int(FA.ntok.median())} tokens | hit cap: {100*(FA.ntok>=FD_MAXLEN).mean():.1f}%", flush=True)
    rows.append(analyse(FA, "Ar-APT balanced 400/400"))
else:
    FA = None
    print("\n[4/6] Ar-APT scores not attached — cross-corpus comparison skipped", flush=True)


[4/6] Ar-APT: 800 scored | median 214 words, 315 tokens | hit cap: 0.0%

[3/6] === Ar-APT balanced 400/400 (n=800) ===
[3/6]   corr(d, sqrt(len))     pearson -0.401 (p=2.6e-32) | spearman -0.394
[3/6]   variance of d explained by length: 16.1%
[3/6]   AUC  d (raw)                =  35.34
[3/6]   AUC  length alone           =  66.22
[3/6]   AUC  d after removing length=  47.44   (change +12.10)
[3/6]   class means of d: human -4.710 | AI -7.351
[3/6]   class median tokens: human 274 | AI 336


## 5 · Verdict

In [5]:
R = pd.DataFrame(rows)
print("\n[5/6] SUMMARY", flush=True)
print(R.to_string(index=False), flush=True)

o = R[R.corpus.str.startswith("OUR corpus (test")].iloc[0]
print("\n[5/6] READING:", flush=True)
strong_len = abs(o.corr_sqrt) > 0.3 or o.auc_len > 60
recovers   = (o.auc_resid - o.auc_d) > 5
if strong_len and recovers:
    print("[5/6]   H1 CONFIRMED — d largely tracks LENGTH, and removing it recovers real signal.", flush=True)
    print("[5/6]   Thesis line: the statistic is not length-normalised; on a corpus with variable", flush=True)
    print("[5/6]   article length it measures length, not authorship.", flush=True)
elif strong_len:
    print("[5/6]   PARTIAL H1 — d is length-driven, but nothing is recovered once length is removed:", flush=True)
    print("[5/6]   length explains the SIGN REVERSAL across corpora, while the surrogate supplies", flush=True)
    print("[5/6]   no authorship signal of its own.", flush=True)
else:
    print("[5/6]   H2 CONFIRMED — length is not the driver. AraGPT2 as a black-box surrogate simply", flush=True)
    print("[5/6]   fails to separate these generators, which is the reportable negative result.", flush=True)
if len(R[R.corpus.str.startswith("Ar-APT")]):
    a = R[R.corpus.str.startswith("Ar-APT")].iloc[0]
    print(f"\n[5/6]   cross-corpus: our median tokens {o.med_tok_human:.0f}/{o.med_tok_ai:.0f} (h/AI) vs "
          f"Ar-APT {a.med_tok_human:.0f}/{a.med_tok_ai:.0f}", flush=True)
    print(f"[5/6]   d means ours {o.mean_d_human:+.2f}/{o.mean_d_ai:+.2f} vs "
          f"Ar-APT {a.mean_d_human:+.2f}/{a.mean_d_ai:+.2f}  -> the sign flip lives here", flush=True)


[5/6] SUMMARY
                 corpus    n  corr_sqrt  var_expl_pct  auc_d  auc_len  auc_resid  mean_d_human  mean_d_ai  med_tok_human  med_tok_ai
OUR corpus (all splits) 7101     -0.185           3.4  51.64    47.97      51.38        -5.085     -3.756          801.0       779.0
OUR corpus (test split) 1093     -0.195           3.8  50.44    48.84      50.15        -4.875     -3.372          759.0       756.5
Ar-APT balanced 400/400  800     -0.401          16.1  35.34    66.22      47.44        -4.710     -7.351          273.5       336.5

[5/6] READING:
[5/6]   H2 CONFIRMED — length is not the driver. AraGPT2 as a black-box surrogate simply
[5/6]   fails to separate these generators, which is the reportable negative result.

[5/6]   cross-corpus: our median tokens 759/756 (h/AI) vs Ar-APT 274/336
[5/6]   d means ours -4.88/-3.37 vs Ar-APT -4.71/-7.35  -> the sign flip lives here


## 6 · Save

In [6]:
R.to_parquet(f"{OUT}/nb18_fastdetect_length_diagnosis.parquet", index=False)
FD[["article_id","label","split","fastdetect","words","ntok","capped"]] \
  .to_parquet(f"{OUT}/nb18_fd_ours_with_length.parquet", index=False)
if FA is not None:
    FA[["arapt_id","label","fastdetect","words","ntok"]] \
      .to_parquet(f"{OUT}/nb18_fd_arapt_with_length.parquet", index=False)
print("[6/6] saved nb18_*.parquet", flush=True)
print("[6/6] done — this turns 'Fast-DetectGPT failed' into a mechanism you can defend.", flush=True)

[6/6] saved nb18_*.parquet
[6/6] done — this turns 'Fast-DetectGPT failed' into a mechanism you can defend.
